# RAGTruth Dataset Statistics & Visualization

**Project:** Multi-Agent Hallucination Detection in RAG Systems  
**Dataset:** RAGTruth (Wu et al., 2023)  

This notebook visualizes the dataset used for training and evaluation.

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import numpy as np
from collections import Counter
from pathlib import Path

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 12

SPLITS_DIR = Path('splits')

def load_jsonl(path):
    records = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

train = load_jsonl(SPLITS_DIR / 'train.jsonl')
test_full = load_jsonl(SPLITS_DIR / 'test_full.jsonl')
test_bal = load_jsonl(SPLITS_DIR / 'test_balanced.jsonl')
all_data = train + test_full

print(f'Train:          {len(train):>6} records')
print(f'Test (full):    {len(test_full):>6} records')
print(f'Test (balanced):{len(test_bal):>6} records')
print(f'Total:          {len(all_data):>6} records')

## 1. Split Size Table

In [ ]:
split_df = pd.DataFrame([
    {'Split': 'Training', 'Records': len(train), 'Percentage': f'{len(train)/len(all_data)*100:.1f}%', 'Used For': 'Official LLaMA-2-13B baseline only'},
    {'Split': 'Test (full)', 'Records': len(test_full), 'Percentage': f'{len(test_full)/len(all_data)*100:.1f}%', 'Used For': 'Full evaluation set'},
    {'Split': 'Test (balanced)', 'Records': len(test_bal), 'Percentage': f'{len(test_bal)/len(all_data)*100:.1f}%', 'Used For': 'Primary eval (200/task)'},
    {'Split': 'Total', 'Records': len(all_data), 'Percentage': '100.0%', 'Used For': ''},
])
print('Table 1: Dataset Splits')
print(split_df.to_string(index=False))

## 2. Task Type Distribution

In [ ]:
task_counts = Counter(r['task_type'] for r in all_data)
tasks = ['QA', 'Summary', 'Data2txt']
counts = [task_counts[t] for t in tasks]
colors = ['#4C72B0', '#DD8452', '#55A868']

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(tasks, counts, color=colors, edgecolor='white', linewidth=1.5)
for bar, count in zip(bars, counts):
    pct = count / len(all_data) * 100
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
            f'{count:,}\n({pct:.1f}%)', ha='center', fontsize=11)
ax.set_ylabel('Number of Records')
ax.set_title('Figure 1: Task Type Distribution (Full Dataset)')
ax.set_ylim(0, max(counts) * 1.2)
plt.tight_layout()
plt.savefig('figures/fig1_task_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: figures/fig1_task_distribution.png')

## 3. Hallucination Rate by Task

In [ ]:
halluc_rates = {}
for task in tasks:
    subset = [r for r in test_full if r['task_type'] == task]
    halluc = sum(1 for r in subset if len(r.get('labels', [])) > 0)
    halluc_rates[task] = (halluc / len(subset) * 100, halluc, len(subset))

fig, ax = plt.subplots(figsize=(8, 5))
rates = [halluc_rates[t][0] for t in tasks]
bars = ax.bar(tasks, rates, color=['#4C72B0', '#DD8452', '#C44E52'], edgecolor='white', linewidth=1.5)
for bar, task in zip(bars, tasks):
    h, n, total = halluc_rates[task]
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{h:.1f}%\n({n}/{total})', ha='center', fontsize=11)
ax.set_ylabel('Hallucination Rate (%)')
ax.set_title('Figure 2: Hallucination Rate by Task Type (Test Set)')
ax.set_ylim(0, 80)
plt.tight_layout()
plt.savefig('figures/fig2_halluc_rate.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Label Type Breakdown (Evident/Subtle × Baseless/Conflict)

In [ ]:
label_types = ['Evident Conflict', 'Evident Baseless Info', 'Subtle Conflict', 'Subtle Baseless Info']
label_counts_per_task = {task: Counter() for task in tasks}

for r in test_full:
    task = r.get('task_type', '')
    for label in r.get('labels', []):
        lt = label.get('label_type', '')
        if lt in label_types:
            label_counts_per_task[task][lt] += 1

x = np.arange(len(tasks))
width = 0.2
label_colors = ['#C44E52', '#4C72B0', '#937860', '#55A868']

fig, ax = plt.subplots(figsize=(11, 6))
for i, (lt, color) in enumerate(zip(label_types, label_colors)):
    vals = [label_counts_per_task[t][lt] for t in tasks]
    ax.bar(x + i * width, vals, width, label=lt, color=color, edgecolor='white')

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(tasks)
ax.set_ylabel('Count')
ax.set_title('Figure 3: Hallucination Type Distribution by Task (Test Set)')
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('figures/fig3_label_types.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Response Length Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
for ax, task, color in zip(axes, tasks, colors):
    subset = [r for r in all_data if r['task_type'] == task]
    lengths = [len(r['response'].split()) for r in subset]
    ax.hist(lengths, bins=30, color=color, edgecolor='white', alpha=0.85)
    ax.axvline(np.mean(lengths), color='red', linestyle='--', linewidth=2,
               label=f'Mean: {np.mean(lengths):.0f}')
    ax.set_title(f'{task}')
    ax.set_xlabel('Word Count')
    ax.set_ylabel('Frequency')
    ax.legend(fontsize=10)
fig.suptitle('Figure 4: Response Length Distribution by Task', fontsize=14)
plt.tight_layout()
plt.savefig('figures/fig4_response_length.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Model Distribution

In [ ]:
model_counts = Counter(r['model'] for r in all_data)
models = list(model_counts.keys())
mcounts = [model_counts[m] for m in models]

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(models, mcounts, color='#4C72B0', edgecolor='white')
for i, v in enumerate(mcounts):
    ax.text(v + 50, i, str(v), va='center', fontsize=11)
ax.set_xlabel('Number of Responses')
ax.set_title('Figure 5: LLM Model Distribution')
plt.tight_layout()
plt.savefig('figures/fig5_model_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Balanced Test Set Verification

In [ ]:
bal_task_counts = Counter(r['task_type'] for r in test_bal)
bal_halluc = Counter()
for r in test_bal:
    task = r['task_type']
    if len(r.get('labels', [])) > 0:
        bal_halluc[task] += 1

print('Balanced Test Set (test_balanced.jsonl):')
print(f'  Total: {len(test_bal)} records')
print()
bal_df = pd.DataFrame([
    {
        'Task': task,
        'Total': bal_task_counts[task],
        'Hallucinated': bal_halluc[task],
        'Clean': bal_task_counts[task] - bal_halluc[task],
        'Halluc Rate': f"{bal_halluc[task]/bal_task_counts[task]*100:.1f}%"
    }
    for task in tasks
])
print(bal_df.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Task balance
axes[0].pie([bal_task_counts[t] for t in tasks], labels=tasks,
            colors=colors, autopct='%1.0f%%', startangle=90)
axes[0].set_title('Task Balance (should be equal thirds)')

# Hallucination balance
halluc_counts = [bal_halluc[t] for t in tasks]
clean_counts = [bal_task_counts[t] - bal_halluc[t] for t in tasks]
x = np.arange(len(tasks))
axes[1].bar(tasks, halluc_counts, label='Hallucinated', color='#C44E52')
axes[1].bar(tasks, clean_counts, bottom=halluc_counts, label='Clean', color='#55A868')
axes[1].set_title('Hallucination Balance per Task')
axes[1].legend()
axes[1].set_ylabel('Count')

fig.suptitle('Figure 6: Balanced Test Set Composition (n=600)', fontsize=14)
plt.tight_layout()
plt.savefig('figures/fig6_balanced_test.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Sample Data Cards

In [ ]:
print('=' * 80)
print('SAMPLE DATA CARDS (1 per task, from test set)')
print('=' * 80)

for task in tasks:
    # Find a hallucinated example
    examples = [r for r in test_full if r['task_type'] == task and len(r.get('labels', [])) > 0]
    if not examples:
        continue
    ex = examples[0]
    
    print(f'\n--- Task: {task} | Model: {ex["model"]} ---')
    ref = str(ex.get('reference', ''))[:300]
    print(f'Reference (truncated): {ref}...')
    print(f'Response: {ex["response"][:400]}...')
    print(f'Hallucination Labels ({len(ex["labels"])} span(s)):')
    for label in ex['labels'][:3]:
        print(f'  [{label["label_type"]}] "{label["text"][:80]}"')
    print()